# 11 · External validation (2025–26), packaging and deployment checks

Score the frozen ensemble on data it has never seen, then turn it into a deployable package and verify every hop.

**Key results — 617 origins, 4,319 forecast pairs (Jan 2025 – Nov 2026)**

| Model | MAE (MU) | RMSE | Bias |
|---|---:|---:|---:|
| Weekly naive | 9.94 | 13.83 | +0.09 |
| TCN | 9.02 | 11.42 | −6.33 |
| Chronos-2 calendar LoRA | 7.72 | 11.08 | +0.61 |
| **Frozen 50/50 ensemble** | **7.48** | **9.90** | −2.86 |

Then: write `src/gridpulse/` (`tcn_model.py`, `inference.py`), check parity with the notebook (max difference < 1e-5 MU), build the model bundle, test it in a clean subprocess, publish it to Hugging Face, and verify the live Modal endpoint and the standalone client return identical forecasts.

> Weekly-naive row computed with `scripts/evaluate_baselines.py` on the same pairs.

> Paths are relative to `notebooks/`. Outputs are from the original research run; see [`data/README.md`](../data/README.md) for data that must be downloaded.

➡ Next: [12 · Live forecasting prototype](12_live_forecasting_prototype.ipynb)

Load external data

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

DATA = Path("../data/processed")

external = pd.read_csv(
    DATA / "gridpulse_v2_external_test.csv",
    parse_dates=["date"]
).sort_values("date")

print("Shape:", external.shape)
print("Columns:", external.columns.tolist())
print("Date range:", external["date"].min(), "→", external["date"].max())

display(external.head())
display(external.tail())

print("\nMissing values:")
display(external.isna().sum())

Shape: (687, 4)
Columns: ['date', 'energy_met_mu', 'max_demand_met_mw', 'energy_missing']
Date range: 2025-01-01 00:00:00 → 2026-11-18 00:00:00


,date,energy_met_mu,max_demand_met_mw,energy_missing
0,2025-01-01,87.43,5519.0,False
1,2025-01-02,86.74,5155.0,False
2,2025-01-03,89.69,5611.0,False
3,2025-01-04,84.23,5065.0,False
4,2025-01-05,81.57,5115.0,False


,date,energy_met_mu,max_demand_met_mw,energy_missing
682,2026-11-14,79.90,4142.0,False
683,2026-11-15,74.85,3819.0,False
684,2026-11-16,78.75,4091.0,False
685,2026-11-17,78.62,4081.0,False
686,2026-11-18,78.90,4103.0,False



Missing values:


date                 0
energy_met_mu        2
max_demand_met_mw    2
energy_missing       0
dtype: int64

## Archived GFS forecasts for 2025–2026

In [2]:
gfs = pd.read_parquet(
    DATA / "gridpulse_gfs_previous_runs_2021_2024.parquet"
)

gfs["time"] = pd.to_datetime(gfs["time"])

print("External dates:")
print(external["date"].min(), "→", external["date"].max())

print("\nArchived GFS forecast dates:")
print(gfs["time"].min(), "→", gfs["time"].max())

External dates:
2025-01-01 00:00:00 → 2026-11-18 00:00:00

Archived GFS forecast dates:
2021-04-01 00:00:00 → 2024-12-31 23:00:00


In [3]:
import requests
import pandas as pd
from pathlib import Path

url = "https://previous-runs-api.open-meteo.com/v1/forecast"

frames = []

for start in pd.date_range("2025-01-01", "2026-11-01", freq="MS"):
    end = min(
        start + pd.offsets.MonthEnd(0),
        pd.Timestamp("2026-11-18")
    )

    params = {
        "latitude": 28.6139,
        "longitude": 77.2090,
        "start_date": start.strftime("%Y-%m-%d"),
        "end_date": end.strftime("%Y-%m-%d"),
        "hourly": ",".join(
            f"temperature_2m_previous_day{i}"
            for i in range(1, 8)
        ),
        "models": "gfs_global",
        "timezone": "Asia/Kolkata",
    }

    response = requests.get(url, params=params, timeout=90)
    response.raise_for_status()

    frame = pd.DataFrame(response.json()["hourly"])
    frames.append(frame)

    print(start.strftime("%Y-%m"), len(frame), "hours")

gfs_external = pd.concat(frames, ignore_index=True)
gfs_external["time"] = pd.to_datetime(gfs_external["time"])

assert not gfs_external["time"].duplicated().any()

path = DATA / "gridpulse_gfs_previous_runs_2025_2026.parquet"
gfs_external.to_parquet(path, index=False)

print("\nSaved:", path)
print("Total hours:", len(gfs_external))

2025-01 744 hours
2025-02 672 hours
2025-03 744 hours
2025-04 720 hours
2025-05 744 hours
2025-06 720 hours
2025-07 744 hours
2025-08 744 hours
2025-09 720 hours
2025-10 744 hours
2025-11 720 hours
2025-12 744 hours
2026-01 744 hours
2026-02 672 hours
2026-03 744 hours
2026-04 720 hours
2026-05 744 hours
2026-06 720 hours
2026-07 744 hours
2026-08 744 hours
2026-09 720 hours
2026-10 744 hours
2026-11 432 hours

Saved: ..\data\processed\gridpulse_gfs_previous_runs_2025_2026.parquet
Total hours: 16488


In [4]:
lead_cols = [
    f"temperature_2m_previous_day{i}"
    for i in range(1, 8)
]

coverage = (
    gfs_external
    .assign(month=gfs_external["time"].dt.to_period("M"))
    .groupby("month")[lead_cols]
    .agg(lambda s: 100 * s.notna().mean())
)

display(coverage.round(1))

print("\nMinimum coverage:", coverage.min().min().round(2), "%")

,temperature_2m_previous_day1,temperature_2m_previous_day2,temperature_2m_previous_day3,temperature_2m_previous_day4,temperature_2m_previous_day5,temperature_2m_previous_day6,temperature_2m_previous_day7
month,,,,,,,
2025-01,100.0,100.0,100.0,100.0,100.0,100.0,100.0
2025-02,100.0,100.0,100.0,100.0,100.0,100.0,100.0
2025-03,100.0,100.0,100.0,100.0,100.0,100.0,100.0
2025-04,100.0,100.0,100.0,100.0,100.0,100.0,100.0
2025-05,100.0,100.0,100.0,100.0,100.0,100.0,100.0
2025-06,100.0,100.0,100.0,100.0,100.0,100.0,100.0
2025-07,100.0,100.0,100.0,100.0,100.0,100.0,100.0
2025-08,100.0,100.0,100.0,100.0,100.0,100.0,100.0
2025-09,100.0,100.0,100.0,100.0,100.0,100.0,100.0



Minimum coverage: 100.0 %


## Build the 617 external forecast origins

In [5]:
development = pd.read_csv(
    DATA / "gridpulse_v2_development.csv",
    parse_dates=["date"]
)

demand_all = (
    pd.concat([development, external], ignore_index=True)
    .sort_values("date")
)

assert not demand_all["date"].duplicated().any()

demand_series = demand_all.set_index("date")["energy_met_mu"]

# First origin predicts January 1–7, 2025.
# Last origin must have seven known target days.
external_origins = pd.date_range(
    "2024-12-31",
    external["date"].max() - pd.Timedelta(days=7),
    freq="D"
)

print("Combined demand days:", len(demand_all))
print("Candidate origins:", len(external_origins))
print("First origin:", external_origins.min().date())
print("Last origin:", external_origins.max().date())

valid_origins = []

for origin in external_origins:
    past = demand_series.reindex(
        pd.date_range(end=origin, periods=56)
    )
    future = demand_series.reindex(
        pd.date_range(start=origin + pd.Timedelta(days=1), periods=7)
    )

    if past.notna().all() and future.notna().all():
        valid_origins.append(origin)

valid_origins = pd.DatetimeIndex(valid_origins)

print("Valid evaluation origins:", len(valid_origins))
print("Target predictions:", len(valid_origins) * 7)

Combined demand days: 2880
Candidate origins: 681
First origin: 2024-12-31
Last origin: 2026-11-11
Valid evaluation origins: 617
Target predictions: 4319


In [6]:
lead_cols = [
    f"temperature_2m_previous_day{i}"
    for i in range(1, 8)
]

daily_gfs = (
    gfs_external
    .assign(date=gfs_external["time"].dt.normalize())
    .groupby("date")[lead_cols]
    .agg(["mean", "max", "min", "count"])
)

daily_gfs.columns = [
    f"{lead}_{stat}"
    for lead, stat in daily_gfs.columns
]

# Reject any incomplete 24-hour temperature profile
for lead in lead_cols:
    incomplete = daily_gfs[f"{lead}_count"] != 24

    for stat in ["mean", "max", "min"]:
        daily_gfs.loc[incomplete, f"{lead}_{stat}"] = np.nan

print("External daily GFS records:", len(daily_gfs))
print(
    "Complete days (all seven leads):",
    daily_gfs[
        [f"{lead}_count" for lead in lead_cols]
    ].eq(24).all(axis=1).sum()
)

External daily GFS records: 687
Complete days (all seven leads): 687


## TCN on the external period (frozen checkpoint)

In [7]:
import torch
import numpy as np

MODEL_DIR = Path("../models/gridpulse_ensemble_v1")

checkpoint = torch.load(
    MODEL_DIR / "tcn_checkpoint.pt",
    map_location="cpu",
    weights_only=True
)

norm = checkpoint["normalization"]

demand_mean = norm["demand_mean"]
demand_std = norm["demand_std"]
temp_mean = np.array(norm["temp_mean"], dtype=np.float32)
temp_std = np.array(norm["temp_std"], dtype=np.float32)

print("Demand mean/std:", demand_mean, demand_std)
print("Temperature means:", temp_mean)
print("Temperature stds:", temp_std)

Demand mean/std: 93.25101470947266 25.11018180847168
Temperature means: [30.172737 34.6845   25.97515 ]
Temperature stds: [6.804233  6.8501096 6.698291 ]


In [8]:
def make_external_sample(origin):
    past_dates = pd.date_range(end=origin, periods=56)
    future_dates = pd.date_range(
        start=origin + pd.Timedelta(days=1),
        periods=7
    )

    history = demand_series.reindex(past_dates).to_numpy(dtype=np.float32)
    actual = demand_series.reindex(future_dates).to_numpy(dtype=np.float32)

    future = []

    for horizon, date in enumerate(future_dates, start=1):
        lead = f"temperature_2m_previous_day{horizon}"
        row = daily_gfs.loc[date]

        future.append([
            row[f"{lead}_mean"],
            row[f"{lead}_max"],
            row[f"{lead}_min"],
            np.sin(2 * np.pi * date.dayofyear / 365.25),
            np.cos(2 * np.pi * date.dayofyear / 365.25),
            np.sin(2 * np.pi * date.dayofweek / 7),
            np.cos(2 * np.pi * date.dayofweek / 7),
        ])

    return (
        history,
        np.asarray(future, dtype=np.float32),
        actual
    )


samples = [make_external_sample(o) for o in valid_origins]

Xh = np.stack([s[0] for s in samples])
Xf = np.stack([s[1] for s in samples])
y_external = np.stack([s[2] for s in samples])

# Apply SAVED training normalization
Xh_n = (Xh - demand_mean) / demand_std

temp = Xf[:, :, :3]
mask = np.isfinite(temp).astype(np.float32)

temp_n = np.where(
    mask.astype(bool),
    (temp - temp_mean) / temp_std,
    0.0
).astype(np.float32)

Xf_n = np.concatenate(
    [temp_n, Xf[:, :, 3:], mask],
    axis=2
)

print("History:", Xh_n.shape)
print("Future features:", Xf_n.shape)
print("Targets:", y_external.shape)

assert Xh_n.shape[1] == 56
assert Xf_n.shape[1:] == (7, 10)
assert np.isfinite(Xh_n).all()
assert np.isfinite(Xf_n).all()

History: (617, 56)
Future features: (617, 7, 10)
Targets: (617, 7)


In [10]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class TemporalBlock(nn.Module):
    def __init__(self, channels, dilation):
        super().__init__()

        self.conv = nn.Conv1d(
            channels, channels,
            kernel_size=3,
            dilation=dilation,
            padding=2 * dilation
        )
        self.norm = nn.GroupNorm(4, channels)
        self.dropout = nn.Dropout(0.1)

    def forward(self, x):
        residual = x
        x = self.conv(x)
        x = x[:, :, :-self.conv.padding[0]]
        x = F.relu(self.norm(x))
        return residual + self.dropout(x)


class GridPulseTCN(nn.Module):
    def __init__(self, future_features=10, hidden=32):
        super().__init__()

        self.input_proj = nn.Conv1d(1, hidden, 1)

        self.tcn = nn.Sequential(
            TemporalBlock(hidden, dilation=1),
            TemporalBlock(hidden, dilation=2),
            TemporalBlock(hidden, dilation=4),
            TemporalBlock(hidden, dilation=8),
            TemporalBlock(hidden, dilation=16),
        )

        self.future_encoder = nn.Sequential(
            nn.Linear(future_features * 7, 32),
            nn.ReLU(),
        )

        self.head = nn.Sequential(
            nn.Linear(hidden + 32, 64),
            nn.ReLU(),
            nn.Dropout(0.1),
            nn.Linear(64, 7)
        )

    def forward(self, history, future):
        x = history.unsqueeze(1)
        x = self.input_proj(x)
        x = self.tcn(x)

        history_features = x[:, :, -1]
        future_features = self.future_encoder(
            future.flatten(start_dim=1)
        )

        combined = torch.cat(
            [history_features, future_features],
            dim=1
        )

        return self.head(combined)

In [11]:
from sklearn.metrics import mean_absolute_error, mean_squared_error

arch = checkpoint["architecture"]

tcn = GridPulseTCN(
    future_features=arch["future_features"],
    hidden=arch["hidden"]
)

tcn.load_state_dict(checkpoint["model_state_dict"])
tcn.eval()

with torch.inference_mode():
    predictions_normalized = tcn(
        torch.from_numpy(Xh_n).float(),
        torch.from_numpy(Xf_n).float()
    ).numpy()

tcn_predictions = (
    predictions_normalized * demand_std + demand_mean
)

assert tcn_predictions.shape == y_external.shape

print("External TCN MAE:",
      round(mean_absolute_error(
          y_external.ravel(),
          tcn_predictions.ravel()
      ), 4))

print("External TCN RMSE:",
      round(np.sqrt(mean_squared_error(
          y_external.ravel(),
          tcn_predictions.ravel()
      )), 4))

External TCN MAE: 9.0218
External TCN RMSE: 11.424


In [12]:
tcn_external_results = pd.DataFrame({
    "origin": np.repeat(valid_origins, 7),
    "horizon": np.tile(np.arange(1, 8), len(valid_origins)),
    "target_date": np.concatenate([
        pd.date_range(o + pd.Timedelta(days=1), periods=7)
        for o in valid_origins
    ]),
    "actual_mu": y_external.ravel(),
    "tcn_pred": tcn_predictions.ravel()
})

output_path = MODEL_DIR / "external_tcn_predictions.parquet"

tcn_external_results.to_parquet(output_path, index=False)

print("Saved:", output_path)
print("Predictions:", len(tcn_external_results))

Saved: ..\models\gridpulse_ensemble_v1\external_tcn_predictions.parquet
Predictions: 4319


## Load the Chronos-2 calendar LoRA adapter

In [13]:
from pathlib import Path

search_roots = [
    Path("../models"),
    Path("../data"),
]

for root in search_roots:
    if not root.exists():
        continue

    for config in root.rglob("adapter_config.json"):
        print("Adapter:", config.parent)
        print(
            "Weights exist:",
            (config.parent / "adapter_model.safetensors").exists()
        )

Adapter: ..\models\chronos2_cpu_benchmark\finetuned-ckpt
Weights exist: True
Adapter: ..\models\chronos2_forecast_weather_200\finetuned-ckpt
Weights exist: True


In [14]:
from pathlib import Path

roots = [
    Path(".."),
    Path.home() / "Downloads",
]

for root in roots:
    if not root.exists():
        continue

    for path in root.rglob("*"):
        if (
            path.is_file()
            and (
                path.name == "gridpulse_export_v1.zip"
                or path.name == "adapter_config.json"
            )
        ):
            print(path)

..\data\gridpulse_export_v1.zip
..\models\chronos2_cpu_benchmark\finetuned-ckpt\adapter_config.json
..\models\chronos2_forecast_weather_200\finetuned-ckpt\adapter_config.json


In [15]:
import zipfile
from pathlib import Path

zip_path = Path("../data/gridpulse_export_v1.zip")
adapter_dir = Path("../models/chronos2_calendar_200")
adapter_dir.mkdir(parents=True, exist_ok=True)

with zipfile.ZipFile(zip_path) as z:
    calendar_files = [
        name for name in z.namelist()
        if "lora_calendar_200/" in name
        and not name.endswith("/")
    ]

    print("Calendar adapter files:")
    for name in calendar_files:
        print(" ", name)

    assert calendar_files, "Calendar LoRA not found in ZIP"

    for name in calendar_files:
        relative = name.split("lora_calendar_200/", 1)[1]
        destination = adapter_dir / relative
        destination.parent.mkdir(parents=True, exist_ok=True)
        destination.write_bytes(z.read(name))

print("\nExtracted to:", adapter_dir)

assert (adapter_dir / "adapter_config.json").exists()
assert (adapter_dir / "adapter_model.safetensors").exists()

Calendar adapter files:
  lora_calendar_200/README.md
  lora_calendar_200/adapter_config.json
  lora_calendar_200/adapter_model.safetensors

Extracted to: ..\models\chronos2_calendar_200


### Working around PEFT detection

`transformers` caches `is_peft_available()`. If `peft` is installed after the kernel starts, loading the adapter raises an `ImportError`; clearing the cache fixes it. The same workaround lives in `deploy/modal_inference.py`.

In [18]:
import peft
from transformers.utils import is_peft_available

print("PEFT version:", peft.__version__)
print("Transformers sees PEFT:", is_peft_available())

PEFT version: 0.22.0
Transformers sees PEFT: False


In [20]:
import sys
import importlib.metadata as md
import peft
import transformers
import chronos.chronos2.pipeline as cp
from transformers.utils.import_utils import is_peft_available

print("Python:", sys.executable)
print("PEFT version:", peft.__version__)
print("PEFT package metadata:", md.version("peft"))
print("Transformers:", transformers.__version__)

print("\nDependency checks:")
print("Transformers:", is_peft_available())
print("Chronos-2:", cp.is_peft_available())

print("\nSame function object:",
      cp.is_peft_available is is_peft_available)

Python: .venv\Scripts\python.exe
PEFT version: 0.22.0
PEFT package metadata: 0.22.0
Transformers: 5.23.0

Dependency checks:
Transformers: False
Chronos-2: False

Same function object: True


In [22]:
import inspect
import transformers.utils.import_utils as iu

print("PEFT availability check:")
print(inspect.getsource(iu.is_peft_available))

print("\nPackage detection function:")
print(inspect.getsource(iu._is_package_available))

PEFT availability check:
@lru_cache
@_make_compile_constant
def is_peft_available() -> bool:
    return _is_package_available("peft")[0]


Package detection function:
def _is_package_available(pkg_name: str, return_version: bool = False) -> tuple[bool, str]:
    """Check if `pkg_name` exist, and optionally try to get its version"""
    spec = importlib.util.find_spec(pkg_name)
    package_exists = spec is not None
    package_version = "N/A"
    if package_exists and return_version:
        try:
            # Most packages are distributed under their import name. Try it before building the mapping, which
            # stats every file of each distribution without a `top_level.txt`, `torch` >= 2.14 among them
            package_version = importlib.metadata.version(pkg_name)
        except importlib.metadata.PackageNotFoundError:
            for distribution_name in _candidate_distribution_names(pkg_name):
                try:
                    package_version = importlib.metadata.ver

In [23]:
import transformers.utils.import_utils as iu
import chronos.chronos2.pipeline as cp

print("Before:", iu.is_peft_available())

iu.is_peft_available.cache_clear()

print("After:", iu.is_peft_available())
print("Chronos-2:", cp.is_peft_available())

Before: False
After: True
Chronos-2: True


In [24]:
import torch
from chronos import Chronos2Pipeline

chronos = Chronos2Pipeline.from_pretrained(
    "../models/chronos2_calendar_200",
    device_map="cpu",
    torch_dtype=torch.float32,
)

chronos.model.eval()
print("Adapter loaded successfully")

Adapter loaded successfully


## Check: does the calendar-only input contract reproduce the 2024 validation predictions?

In [26]:
import requests
import pandas as pd
import numpy as np

weather_cols = [
    "temperature_2m_max",
    "temperature_2m_min",
    "temperature_2m_mean",
    "relative_humidity_2m_mean",
    "apparent_temperature_max",
    "precipitation_sum",
]

params = {
    "latitude": 28.6139,
    "longitude": 77.2090,
    "start_date": "2024-01-01",
    "end_date": "2026-11-18",
    "daily": ",".join(weather_cols),
    "timezone": "Asia/Kolkata",
}

response = requests.get(
    "https://archive-api.open-meteo.com/v1/archive",
    params=params,
    timeout=120
)
response.raise_for_status()

weather_history = pd.DataFrame(
    response.json()["daily"]
).rename(columns={"time": "date"})

weather_history["date"] = pd.to_datetime(
    weather_history["date"]
)

weather_history = (
    weather_history
    .set_index("date")
    .sort_index()
)

print("Weather range:",
      weather_history.index.min(),
      weather_history.index.max())

print("\nMissing values:")
print(weather_history[weather_cols].isna().sum())

assert weather_history.index.is_unique

Weather range: 2024-01-01 00:00:00 2026-11-18 00:00:00

Missing values:
temperature_2m_max           0
temperature_2m_min           0
temperature_2m_mean          0
relative_humidity_2m_mean    0
apparent_temperature_max     0
precipitation_sum            0
dtype: int64


In [27]:
def calendar_features(dates):
    doy = dates.dayofyear.to_numpy()
    dow = dates.dayofweek.to_numpy()

    return {
        "year_sin": np.sin(2 * np.pi * doy / 365.25).astype(np.float32),
        "year_cos": np.cos(2 * np.pi * doy / 365.25).astype(np.float32),
        "week_sin": np.sin(2 * np.pi * dow / 7).astype(np.float32),
        "week_cos": np.cos(2 * np.pi * dow / 7).astype(np.float32),
    }


def make_chronos_input(origin):
    past_dates = pd.date_range(
        end=origin,
        periods=56,
        freq="D"
    )

    future_dates = pd.date_range(
        start=origin + pd.Timedelta(days=1),
        periods=7,
        freq="D"
    )

    target = demand_series.reindex(
        past_dates
    ).to_numpy(dtype=np.float32)

    past_weather = weather_history.reindex(past_dates)

    past_covariates = {
        col: past_weather[col].to_numpy(dtype=np.float32)
        for col in weather_cols
    }

    past_covariates.update(calendar_features(past_dates))

    future_covariates = calendar_features(future_dates)

    assert target.shape == (56,)
    assert len(past_covariates) == 10
    assert len(future_covariates) == 4

    assert np.isfinite(target).all()
    assert all(
        np.isfinite(values).all()
        for values in past_covariates.values()
    )
    assert all(
        np.isfinite(values).all()
        for values in future_covariates.values()
    )

    return {
        "target": target,
        "past_covariates": past_covariates,
        "future_covariates": future_covariates,
    }


# Validate one external input
sample = make_chronos_input(valid_origins[0])

print("Historical demand:", sample["target"].shape)
print("Past covariates:", list(sample["past_covariates"]))
print("Future covariates:", list(sample["future_covariates"]))

Historical demand: (56,)
Past covariates: ['temperature_2m_max', 'temperature_2m_min', 'temperature_2m_mean', 'relative_humidity_2m_mean', 'apparent_temperature_max', 'precipitation_sum', 'year_sin', 'year_cos', 'week_sin', 'week_cos']
Future covariates: ['year_sin', 'year_cos', 'week_sin', 'week_cos']


In [28]:
origin = pd.Timestamp("2024-10-26")

# Reconstruct input using our verified function
replay_input = make_chronos_input(origin)

chronos.model.eval()

with torch.inference_mode():
    result = chronos.predict(
        inputs=[replay_input],
        prediction_length=7
    )

median_idx = chronos.quantiles.index(0.5)
replayed = np.asarray(result[0])[0, median_idx, :]

# Original frozen 2024 benchmark
benchmark = pd.read_parquet(
    DATA / "gridpulse_chronos2_validation_2024.parquet"
)

benchmark["origin"] = pd.to_datetime(benchmark["origin"])
benchmark["target_date"] = pd.to_datetime(benchmark["target_date"])

reference = (
    benchmark.loc[benchmark["origin"].eq(origin)]
    .sort_values("horizon")
    .copy()
)

assert len(reference) == 7

reference["replayed_pred"] = replayed
reference["difference"] = (
    reference["replayed_pred"]
    - reference["chronos2_lora_calendar_200"]
)

display(reference[
    [
        "target_date",
        "actual_mu",
        "chronos2_lora_calendar_200",
        "replayed_pred",
        "difference"
    ]
].round(4))

print(
    "Maximum absolute difference:",
    round(reference["difference"].abs().max(), 6),
    "MU"
)

,target_date,actual_mu,chronos2_lora_calendar_200,replayed_pred,difference
2100,2024-10-27,88.55,91.1119,91.656601,0.5447
2101,2024-10-28,96.14,102.0179,101.495796,-0.5221
2102,2024-10-29,99.11,105.8111,105.369202,-0.4419
2103,2024-10-30,97.50,103.4137,103.907600,0.4939
2104,2024-10-31,83.06,103.1403,104.689598,1.5493
2105,2024-11-01,76.77,101.3346,103.134102,1.7995
2106,2024-11-02,72.76,95.5161,97.375603,1.8595


Maximum absolute difference: 1.859505 MU


In [29]:
test_origins = pd.to_datetime([
    "2024-03-15",
    "2024-05-15",
    "2024-07-15",
    "2024-10-26",
    "2024-12-15",
])

rows = []

chronos.model.eval()

with torch.inference_mode():
    for origin in test_origins:
        example = make_chronos_input(origin)

        result = chronos.predict(
            inputs=[example],
            prediction_length=7
        )

        median = np.asarray(result[0])[0, median_idx, :]

        reference = (
            benchmark.loc[benchmark["origin"].eq(origin)]
            .sort_values("horizon")
        )

        if len(reference) != 7:
            print("Skipping:", origin.date())
            continue

        difference = (
            median
            - reference["chronos2_lora_calendar_200"].to_numpy()
        )

        rows.append({
            "origin": origin.date(),
            "MAE_difference": np.abs(difference).mean(),
            "max_difference": np.abs(difference).max(),
            "mean_bias": difference.mean(),
        })

display(pd.DataFrame(rows).round(4))

,origin,MAE_difference,max_difference,mean_bias
0,2024-03-15,0.3626,0.6522,0.0453
1,2024-05-15,1.6658,2.0991,-1.6658
2,2024-07-15,2.0230,2.9424,-2.0230
3,2024-10-26,1.0301,1.8595,0.7547
4,2024-12-15,0.4661,0.6320,-0.4661


In [31]:
rows = []
calendar_cols = ["year_sin", "year_cos", "week_sin", "week_cos"]
chronos.model.eval()

with torch.inference_mode():
    for origin in test_origins:
        example = make_chronos_input(origin)

        # Calendar-only historical and future covariates
        example["past_covariates"] = {
            k: v for k, v in example["past_covariates"].items()
            if k in calendar_cols
        }

        result = chronos.predict(
            inputs=[example],
            prediction_length=7
        )

        median = np.asarray(result[0])[0, median_idx, :]

        reference = (
            benchmark.loc[benchmark["origin"].eq(origin)]
            .sort_values("horizon")
        )

        assert len(reference) == 7

        diff = (
            median
            - reference["chronos2_lora_calendar_200"].to_numpy()
        )

        rows.append({
            "origin": origin.date(),
            "mae_difference": np.abs(diff).mean(),
            "max_difference": np.abs(diff).max(),
        })

display(pd.DataFrame(rows).round(4))

,origin,mae_difference,max_difference
0,2024-03-15,0.0024,0.0058
1,2024-05-15,0.0125,0.0308
2,2024-07-15,0.0021,0.0058
3,2024-10-26,0.0105,0.0210
4,2024-12-15,0.0025,0.0041


## Chronos-2 on the external period

In [32]:
from tqdm.auto import tqdm
import numpy as np
import pandas as pd
import torch

chronos.model.eval()
median_idx = chronos.quantiles.index(0.5)

chronos_predictions = []

with torch.inference_mode():
    for origin in tqdm(valid_origins):
        past_dates = pd.date_range(end=origin, periods=56)
        future_dates = pd.date_range(
            start=origin + pd.Timedelta(days=1),
            periods=7
        )

        history = demand_series.reindex(past_dates).to_numpy(
            dtype=np.float32
        )

        assert history.shape == (56,)
        assert np.isfinite(history).all()

        example = {
            "target": history,
            "past_covariates": calendar_features(past_dates),
            "future_covariates": calendar_features(future_dates),
        }

        result = chronos.predict(
            inputs=[example],
            prediction_length=7,
        )

        median = np.asarray(result[0])[0, median_idx, :]

        assert median.shape == (7,)
        assert np.isfinite(median).all()

        chronos_predictions.append(median.copy())

chronos_predictions = np.stack(chronos_predictions)

assert chronos_predictions.shape == (len(valid_origins), 7)
assert chronos_predictions.shape == tcn_predictions.shape
assert chronos_predictions.shape == y_external.shape

print("Chronos predictions:", chronos_predictions.shape)
print("Evaluation mode:", not chronos.model.training)

Chronos predictions: (617, 7)
Evaluation mode: True


## Frozen 50/50 ensemble — external results

In [33]:
ensemble_predictions = (
    0.5 * chronos_predictions
    + 0.5 * tcn_predictions
)

results = []

for name, pred in [
    ("Chronos-2 calendar LoRA", chronos_predictions),
    ("TCN from scratch", tcn_predictions),
    ("Frozen 50/50 ensemble", ensemble_predictions),
]:
    error = pred - y_external

    results.append({
        "Model": name,
        "MAE": np.abs(error).mean(),
        "RMSE": np.sqrt(np.mean(error ** 2)),
        "Bias": error.mean(),
    })

external_results = pd.DataFrame(results)

display(
    external_results.sort_values("MAE").round(4)
)

,Model,MAE,RMSE,Bias
2,Frozen 50/50 ensemble,7.4817,9.8990,-2.8594
0,Chronos-2 calendar LoRA,7.7173,11.0767,0.6132
1,TCN from scratch,9.0218,11.4240,-6.3320


In [34]:
from pathlib import Path
import json
import numpy as np
import pandas as pd

MODEL_DIR = Path("../models/gridpulse_ensemble_v1")
MODEL_DIR.mkdir(parents=True, exist_ok=True)

external_predictions = pd.DataFrame({
    "origin": np.repeat(valid_origins, 7),
    "horizon": np.tile(np.arange(1, 8), len(valid_origins)),
    "target_date": np.concatenate([
        pd.date_range(o + pd.Timedelta(days=1), periods=7)
        for o in valid_origins
    ]),
    "actual_mu": y_external.ravel(),
    "chronos2_pred": chronos_predictions.ravel(),
    "tcn_pred": tcn_predictions.ravel(),
    "ensemble_pred": ensemble_predictions.ravel(),
})

assert len(external_predictions) == len(valid_origins) * 7
assert external_predictions[["origin", "target_date", "horizon"]].duplicated().sum() == 0

external_predictions.to_parquet(
    MODEL_DIR / "external_2025_2026_predictions.parquet",
    index=False
)

external_results.to_csv(
    MODEL_DIR / "external_2025_2026_metrics.csv",
    index=False
)

print("Saved external predictions and metrics.")

Saved external predictions and metrics.


## Package a portable inference module

These cells wrote the files now in `src/gridpulse/`. Re-running them overwrites those files.

In [35]:
from pathlib import Path

PACKAGE_DIR = Path("../src/gridpulse")
PACKAGE_DIR.mkdir(parents=True, exist_ok=True)

(PACKAGE_DIR / "__init__.py").touch()

print("Package directory:", PACKAGE_DIR.resolve())

Package directory: src\gridpulse


In [36]:
%%writefile ../src/gridpulse/tcn_model.py

import torch
import torch.nn as nn
import torch.nn.functional as F


class TemporalBlock(nn.Module):
    def __init__(self, channels, dilation):
        super().__init__()

        self.conv = nn.Conv1d(
            channels,
            channels,
            kernel_size=3,
            dilation=dilation,
            padding=2 * dilation,
        )

        self.norm = nn.GroupNorm(4, channels)
        self.dropout = nn.Dropout(0.1)

    def forward(self, x):
        residual = x
        x = self.conv(x)

        x = x[:, :, :-self.conv.padding[0]]
        x = F.relu(self.norm(x))

        return residual + self.dropout(x)


class GridPulseTCN(nn.Module):
    def __init__(self, future_features=10, hidden=32):
        super().__init__()

        self.input_proj = nn.Conv1d(1, hidden, 1)

        self.tcn = nn.Sequential(
            TemporalBlock(hidden, dilation=1),
            TemporalBlock(hidden, dilation=2),
            TemporalBlock(hidden, dilation=4),
            TemporalBlock(hidden, dilation=8),
            TemporalBlock(hidden, dilation=16),
        )

        self.future_encoder = nn.Sequential(
            nn.Linear(future_features * 7, 32),
            nn.ReLU(),
        )

        self.head = nn.Sequential(
            nn.Linear(hidden + 32, 64),
            nn.ReLU(),
            nn.Dropout(0.1),
            nn.Linear(64, 7),
        )

    def forward(self, history, future):
        x = history.unsqueeze(1)
        x = self.input_proj(x)
        x = self.tcn(x)

        history_features = x[:, :, -1]

        future_features = self.future_encoder(
            future.flatten(start_dim=1)
        )

        combined = torch.cat(
            [history_features, future_features],
            dim=1,
        )

        return self.head(combined)

Writing ../src/gridpulse/tcn_model.py


In [37]:
import sys
import torch

sys.path.insert(0, str(Path("../src").resolve()))

from gridpulse.tcn_model import GridPulseTCN

checkpoint = torch.load(
    "../models/gridpulse_ensemble_v1/tcn_checkpoint.pt",
    map_location="cpu",
    weights_only=True,
)

arch = checkpoint["architecture"]

portable_tcn = GridPulseTCN(
    future_features=arch["future_features"],
    hidden=arch["hidden"],
)

portable_tcn.load_state_dict(
    checkpoint["model_state_dict"],
    strict=True,
)

portable_tcn.eval()

with torch.inference_mode():
    restored_predictions = portable_tcn(
        torch.from_numpy(Xh_n[:1]).float(),
        torch.from_numpy(Xf_n[:1]).float(),
    ).numpy()

original_predictions = (
    (tcn_predictions[:1] - demand_mean) / demand_std
)

difference = abs(
    restored_predictions - original_predictions
).max()

print("Maximum prediction difference:", difference)

assert difference < 1e-4
print("Portable TCN verified.")

Maximum prediction difference: 2.3841858e-07
Portable TCN verified.


In [38]:
%%writefile ../src/gridpulse/inference.py

from pathlib import Path

import numpy as np
import pandas as pd
import torch

from chronos import Chronos2Pipeline
from .tcn_model import GridPulseTCN


CALENDAR_COLS = ("year_sin", "year_cos", "week_sin", "week_cos")


def calendar_features(dates):
    dates = pd.DatetimeIndex(dates)

    doy = dates.dayofyear.to_numpy()
    dow = dates.dayofweek.to_numpy()

    return {
        "year_sin": np.sin(2 * np.pi * doy / 365.25).astype(np.float32),
        "year_cos": np.cos(2 * np.pi * doy / 365.25).astype(np.float32),
        "week_sin": np.sin(2 * np.pi * dow / 7).astype(np.float32),
        "week_cos": np.cos(2 * np.pi * dow / 7).astype(np.float32),
    }


class GridPulseForecaster:

    def __init__(self, models_dir):
        models_dir = Path(models_dir)

        checkpoint = torch.load(
            models_dir / "gridpulse_ensemble_v1" / "tcn_checkpoint.pt",
            map_location="cpu",
            weights_only=True,
        )

        arch = checkpoint["architecture"]
        self.norm = checkpoint["normalization"]

        assert arch["history_length"] == 56
        assert arch["prediction_length"] == 7
        assert arch["future_features"] == 10

        self.tcn = GridPulseTCN(
            future_features=arch["future_features"],
            hidden=arch["hidden"],
        )

        self.tcn.load_state_dict(
            checkpoint["model_state_dict"],
            strict=True,
        )
        self.tcn.eval()

        self.chronos = Chronos2Pipeline.from_pretrained(
            str((models_dir / "chronos2_calendar_200").resolve()),
            device_map="cpu",
            torch_dtype=torch.float32,
        )

        self.chronos.model.eval()
        self.median_idx = self.chronos.quantiles.index(0.5)

    @classmethod
    def from_local(cls, models_dir):
        return cls(models_dir)

    def predict(self, forecast_origin, demand_history_mu, weather_forecast):
        origin = pd.Timestamp(forecast_origin)

        if pd.isna(origin) or origin != origin.normalize():
            raise ValueError("forecast_origin must be a calendar date")

        # Validate demand history.
        history = np.asarray(demand_history_mu, dtype=np.float32)

        if history.shape != (56,):
            raise ValueError("Exactly 56 demand observations are required")

        if not np.isfinite(history).all():
            raise ValueError("Demand history contains invalid values")

        # Validate seven future weather records.
        if len(weather_forecast) != 7:
            raise ValueError("Exactly seven weather records are required")

        future_dates = pd.date_range(
            origin + pd.Timedelta(days=1),
            periods=7,
            freq="D",
        )

        past_dates = pd.date_range(
            end=origin,
            periods=56,
            freq="D",
        )

        # Require chronological weather records.
        temperatures = []

        for expected_date, record in zip(future_dates, weather_forecast):
            supplied_date = pd.Timestamp(record["date"])

            if supplied_date != expected_date:
                raise ValueError(
                    f"Expected weather for {expected_date.date()}, "
                    f"received {supplied_date.date()}"
                )

            temperatures.append([
                record["mean"],
                record["max"],
                record["min"],
            ])

        temperatures = np.asarray(temperatures, dtype=np.float32)

        if not np.isfinite(temperatures).all():
            raise ValueError("Weather forecast contains invalid temperatures")

        # --------------------------------------------------
        # Chronos-2: calendar-only inference contract
        # --------------------------------------------------
        chronos_input = {
            "target": history,
            "past_covariates": calendar_features(past_dates),
            "future_covariates": calendar_features(future_dates),
        }

        self.chronos.model.eval()

        with torch.inference_mode():
            result = self.chronos.predict(
                inputs=[chronos_input],
                prediction_length=7,
            )

        chronos_pred = np.asarray(result[0])[
            0, self.median_idx, :
        ].copy()

        # --------------------------------------------------
        # TCN: saved normalization and feature ordering
        # --------------------------------------------------
        norm = self.norm

        history_n = (
            (history - norm["demand_mean"]) / norm["demand_std"]
        ).astype(np.float32)

        temp_mean = np.asarray(norm["temp_mean"], dtype=np.float32)
        temp_std = np.asarray(norm["temp_std"], dtype=np.float32)

        temp_n = (temperatures - temp_mean) / temp_std

        cal = calendar_features(future_dates)

        calendar_array = np.column_stack([
            cal[col] for col in CALENDAR_COLS
        ]).astype(np.float32)

        # Three normalized temperatures, four calendar
        # features, and three temperature-availability masks.
        future_features = np.concatenate([
            temp_n,
            calendar_array,
            np.ones((7, 3), dtype=np.float32),
        ], axis=1).astype(np.float32)

        assert future_features.shape == (7, 10)

        self.tcn.eval()

        with torch.inference_mode():
            tcn_n = self.tcn(
                torch.from_numpy(history_n[None, :]),
                torch.from_numpy(future_features[None, :, :]),
            ).numpy()[0]

        tcn_pred = (
            tcn_n * norm["demand_std"] + norm["demand_mean"]
        )

        # Frozen ensemble weights.
        ensemble_pred = 0.5 * chronos_pred + 0.5 * tcn_pred

        return {
            "forecast_origin": origin.strftime("%Y-%m-%d"),
            "predictions": [
                {
                    "date": date.strftime("%Y-%m-%d"),
                    "horizon": i + 1,
                    "chronos2_mu": float(chronos_pred[i]),
                    "tcn_mu": float(tcn_pred[i]),
                    "ensemble_mu": float(ensemble_pred[i]),
                }
                for i, date in enumerate(future_dates)
            ],
        }

Writing ../src/gridpulse/inference.py


In [39]:
import importlib
import gridpulse.inference as inference_module

importlib.reload(inference_module)

GridPulseForecaster = inference_module.GridPulseForecaster

forecaster = GridPulseForecaster.from_local("../models")

origin = valid_origins[0]

weather_records = []

for i, date in enumerate(
    pd.date_range(origin + pd.Timedelta(days=1), periods=7)
):
    weather_records.append({
        "date": date.strftime("%Y-%m-%d"),
        "mean": float(Xf[0, i, 0]),
        "max": float(Xf[0, i, 1]),
        "min": float(Xf[0, i, 2]),
    })

result = forecaster.predict(
    forecast_origin=str(origin.date()),
    demand_history_mu=Xh[0].tolist(),
    weather_forecast=weather_records,
)

predictions = pd.DataFrame(result["predictions"])

checks = {
    "Chronos-2": (
        predictions["chronos2_mu"].to_numpy(),
        chronos_predictions[0],
    ),
    "TCN": (
        predictions["tcn_mu"].to_numpy(),
        tcn_predictions[0],
    ),
    "Ensemble": (
        predictions["ensemble_mu"].to_numpy(),
        ensemble_predictions[0],
    ),
}

for name, (new, reference) in checks.items():
    difference = np.max(np.abs(new - reference))
    print(f"{name:12s} max difference: {difference:.6f} MU")

display(predictions.round(2))

Chronos-2    max difference: 0.000000 MU
TCN          max difference: 0.000008 MU
Ensemble     max difference: 0.000008 MU


,date,horizon,chronos2_mu,tcn_mu,ensemble_mu
0,2025-01-01,1,84.63,82.90,83.77
1,2025-01-02,2,84.67,81.14,82.91
2,2025-01-03,3,84.93,79.55,82.24
3,2025-01-04,4,82.81,78.71,80.76
4,2025-01-05,5,81.42,79.43,80.42
5,2025-01-06,6,84.40,79.95,82.18
6,2025-01-07,7,85.19,79.74,82.47


## Build the model bundle and verify it in a clean process

In [40]:
from pathlib import Path
import shutil

ROOT = Path("..").resolve()
PACKAGE = ROOT / "dist" / "gridpulse-ensemble-v1"

adapter_src = ROOT / "models" / "chronos2_calendar_200"
ensemble_src = ROOT / "models" / "gridpulse_ensemble_v1"
code_src = ROOT / "src" / "gridpulse"

(PACKAGE / "chronos2_calendar_200").mkdir(
    parents=True, exist_ok=True
)
(PACKAGE / "gridpulse").mkdir(exist_ok=True)

for filename in ["adapter_config.json", "adapter_model.safetensors"]:
    shutil.copy2(
        adapter_src / filename,
        PACKAGE / "chronos2_calendar_200" / filename
    )

shutil.copy2(
    ensemble_src / "tcn_checkpoint.pt",
    PACKAGE / "tcn_checkpoint.pt"
)

shutil.copy2(
    ensemble_src / "metadata.json",
    PACKAGE / "metadata.json"
)

for filename in ["__init__.py", "tcn_model.py", "inference.py"]:
    shutil.copy2(
        code_src / filename,
        PACKAGE / "gridpulse" / filename
    )

print("Distribution:", PACKAGE)
for path in sorted(PACKAGE.rglob("*")):
    if path.is_file():
        print(path.relative_to(PACKAGE))

Distribution: dist\gridpulse-ensemble-v1
chronos2_calendar_200\adapter_config.json
chronos2_calendar_200\adapter_model.safetensors
gridpulse\__init__.py
gridpulse\inference.py
gridpulse\tcn_model.py
metadata.json
tcn_checkpoint.pt


In [41]:
from pathlib import Path
import shutil

# Move model artifacts into the expected directory structure
(PACKAGE / "gridpulse_ensemble_v1").mkdir(exist_ok=True)

shutil.move(
    str(PACKAGE / "tcn_checkpoint.pt"),
    str(PACKAGE / "gridpulse_ensemble_v1" / "tcn_checkpoint.pt")
)

# Move metadata alongside the checkpoint
shutil.move(
    str(PACKAGE / "metadata.json"),
    str(PACKAGE / "gridpulse_ensemble_v1" / "metadata.json")
)

print("Model artifacts:")
for path in sorted(PACKAGE.rglob("*")):
    if path.is_file():
        print(path.relative_to(PACKAGE))

Model artifacts:
chronos2_calendar_200\adapter_config.json
chronos2_calendar_200\adapter_model.safetensors
gridpulse\__init__.py
gridpulse\inference.py
gridpulse\tcn_model.py
gridpulse_ensemble_v1\metadata.json
gridpulse_ensemble_v1\tcn_checkpoint.pt


In [42]:
import sys
import importlib

sys.path.insert(0, str(PACKAGE.resolve()))

from gridpulse.inference import GridPulseForecaster

packaged = GridPulseForecaster.from_local(PACKAGE)

packaged_result = packaged.predict(
    forecast_origin=str(valid_origins[0].date()),
    demand_history_mu=Xh[0].tolist(),
    weather_forecast=weather_records,
)

packaged_predictions = pd.DataFrame(
    packaged_result["predictions"]
)

difference = np.max(np.abs(
    packaged_predictions["ensemble_mu"].to_numpy()
    - ensemble_predictions[0]
))

print(f"Packaged ensemble difference: {difference:.8f} MU")

assert difference < 0.001
print("Packaged inference verified.")

Packaged ensemble difference: 0.00000763 MU
Packaged inference verified.


In [43]:
from importlib.metadata import version
from pathlib import Path

packages = [
    "torch",
    "numpy",
    "pandas",
    "chronos-forecasting",
    "transformers",
    "peft",
    "accelerate",
    "safetensors",
]

requirements = []

for package in packages:
    try:
        requirements.append(f"{package}=={version(package)}")
    except Exception as exc:
        print(f"Could not resolve {package}: {exc}")

requirements_path = PACKAGE / "requirements.txt"
requirements_path.write_text(
    "\n".join(requirements) + "\n",
    encoding="utf-8"
)

print(requirements_path.read_text())

torch==2.15.0
numpy==2.5.5
pandas==3.1.1
chronos-forecasting==2.3.2
transformers==5.23.0
peft==0.22.0
accelerate==1.16.0
safetensors==0.8.0



In [44]:
readme = """---
license: apache-2.0
tags:
- time-series-forecasting
- electricity
- pytorch
- chronos
- tcn
- ensemble
---

# GridPulse Ensemble v1

Seven-day electricity-supplied forecasting for Delhi, India.

## Architecture

A frozen 50/50 ensemble of:

1. Amazon Chronos-2 with a calendar-only LoRA adapter.
2. A lightweight Temporal Convolutional Network trained from scratch.

The Chronos-2 branch uses historical electricity supplied
and calendar features.

The TCN branch uses historical electricity supplied, future
temperature forecasts, calendar features, and availability masks.

Both models predict the next seven daily electricity-supplied
values in million units (MU).

## Evaluation

| Model | 2024 MAE | 2025–2026 MAE | 2025–2026 RMSE |
|---|---:|---:|---:|
| Chronos-2 calendar LoRA | 6.8693 | 7.7173 | 11.0767 |
| TCN | 7.2733 | 9.0218 | 11.4240 |
| 50/50 Ensemble | 6.0547 | 7.4817 | 9.8990 |

The external evaluation covers 617 rolling forecast origins and
4,319 origin-horizon prediction pairs.

The ensemble weights were selected using 2024 validation data
and frozen before the 2025–2026 evaluation.

## Input contract

- Forecast-origin date
- Exactly 56 chronological daily electricity observations (MU)
- Exactly seven daily temperature forecasts, ordered by target date
- Each temperature forecast contains mean, maximum, and minimum values in °C

The model returns seven daily predictions from each component
and their 50/50 ensemble.

## Limitations

- Intended for Delhi electricity supplied, not unrestricted
  underlying electricity demand.
- Validation uses overlapping rolling forecasts.
- Historical GFS weather forecasts come from Open-Meteo's
  Previous Runs API. Exact forecast issuance and publication
  times were not independently verified.
- Historical backtesting is not equivalent to live deployment.
- Performance may deteriorate under distribution shift.
- This is a research prototype, not an operational grid-control system.

## Reproducibility

The repository contains:

- Chronos-2 LoRA adapter
- TCN checkpoint and saved normalization parameters
- Python inference implementation
- Ensemble metadata
- Dependency manifest

The Chronos-2 base model is loaded separately from
`amazon/chronos-2`.

See `gridpulse/inference.py` for the inference interface.

## Research status

Training and historical external evaluation are complete.
Live operational performance has not been established.
"""

(PACKAGE / "README.md").write_text(
    readme,
    encoding="utf-8"
)

print("Model card saved:", PACKAGE / "README.md")
print("Files:", len(list(PACKAGE.rglob("*"))))

Model card saved: dist\gridpulse-ensemble-v1\README.md
Files: 12


In [46]:
import subprocess
import sys
import json
import os
import textwrap
import numpy as np

# Use the first external test example as our reference.
payload = {
    "origin": str(valid_origins[0].date()),
    "history": Xh[0].tolist(),
    "weather": weather_records,
}

script = textwrap.dedent("""
    import sys
    import json
    import numpy as np
    import torch

    from gridpulse.inference import GridPulseForecaster

    request = json.loads(sys.stdin.read())

    model = GridPulseForecaster.from_local(".")

    result = model.predict(
        forecast_origin=request["origin"],
        demand_history_mu=request["history"],
        weather_forecast=request["weather"],
    )

    predictions = [
        row["ensemble_mu"]
        for row in result["predictions"]
    ]

    print("GRIDPULSE_RESULT=" + json.dumps(predictions))
""")

# Keep the process isolated from the notebook's source directory.
env = os.environ.copy()
env["PYTHONPATH"] = str(PACKAGE.resolve())

process = subprocess.run(
    [sys.executable, "-c", script],
    input=json.dumps(payload),
    text=True,
    capture_output=True,
    cwd=PACKAGE.resolve(),
    env=env,
    timeout=180,
    encoding="utf-8",
    errors="replace",
)

if process.returncode != 0:
    print(process.stderr[-4000:])
    raise RuntimeError("Clean-process test failed")

result_line = next(
    line for line in process.stdout.splitlines()
    if line.startswith("GRIDPULSE_RESULT=")
)

clean_predictions = np.array(
    json.loads(result_line.split("=", 1)[1])
)

difference = np.max(
    np.abs(clean_predictions - ensemble_predictions[0])
)

print("Clean-process maximum difference:", difference, "MU")

assert difference < 0.001
print("GridPulse clean-process test PASSED")

Clean-process maximum difference: 7.62939453125e-06 MU
GridPulse clean-process test PASSED


## Publish to Hugging Face and verify the download

In [47]:
from huggingface_hub import HfApi

api = HfApi()
user = api.whoami()

print("Authenticated as:", user["name"])

Authenticated as: shishirgarde


In [49]:
from huggingface_hub import HfApi

api = HfApi()

repo_id = f"{api.whoami()['name']}/gridpulse-ensemble-v1"

repo = api.create_repo(
    repo_id=repo_id,
    repo_type="model",
    private=True,
    exist_ok=True,
)

print("Repository:", repo)

Repository: https://huggingface.co/shishirgarde/gridpulse-ensemble-v1


In [51]:
from pathlib import Path

repo_id = "shishirgarde/gridpulse-ensemble-v1"

allowed_files = {
    "README.md",
    "requirements.txt",
    "chronos2_calendar_200/adapter_config.json",
    "chronos2_calendar_200/adapter_model.safetensors",
    "gridpulse_ensemble_v1/tcn_checkpoint.pt",
    "gridpulse_ensemble_v1/metadata.json",
    "gridpulse/__init__.py",
    "gridpulse/tcn_model.py",
    "gridpulse/inference.py",
}

actual_files = {
    p.relative_to(PACKAGE).as_posix()
    for p in PACKAGE.rglob("*")
    if p.is_file()
    and "__pycache__" not in p.parts
    and p.suffix != ".pyc"
}

assert actual_files == allowed_files, (
    f"Unexpected files: {actual_files - allowed_files}"
)

print("Manifest verified:", len(actual_files), "files")

print("Files to publish:")
for name in sorted(allowed_files):
    path = PACKAGE / name
    print(f"{name:58s} {path.stat().st_size / 1024:.1f} KB"
          if path.exists() else f"MISSING: {name}")

assert allowed_files <= actual_files, "Required files missing"
assert actual_files == allowed_files, (
    f"Unexpected files: {actual_files - allowed_files}"
)

print("\nManifest verified.")

Manifest verified: 9 files
Files to publish:
README.md                                                  2.5 KB
chronos2_calendar_200/adapter_config.json                  1.3 KB
chronos2_calendar_200/adapter_model.safetensors            4740.8 KB
gridpulse/__init__.py                                      0.0 KB
gridpulse/inference.py                                     6.3 KB
gridpulse/tcn_model.py                                     1.8 KB
gridpulse_ensemble_v1/metadata.json                        0.5 KB
gridpulse_ensemble_v1/tcn_checkpoint.pt                    99.1 KB
requirements.txt                                           0.1 KB

Manifest verified.


In [52]:
from huggingface_hub import HfApi

api = HfApi()

info = api.repo_info(repo_id, repo_type="model")
assert info.private, "Repository must be private for initial upload"

commit = api.upload_folder(
    repo_id=repo_id,
    repo_type="model",
    folder_path=str(PACKAGE.resolve()),
    allow_patterns=sorted(allowed_files),
    commit_message="Upload verified GridPulse Ensemble v1",
)

print("Upload committed.")
print("Commit:", commit.oid)
print("URL:", commit.commit_url)

Upload committed.
Commit: e2dcdfda9380130aa0c2624f4d7c641f0669c2d8
URL: https://huggingface.co/shishirgarde/gridpulse-ensemble-v1/commit/e2dcdfda9380130aa0c2624f4d7c641f0669c2d8


In [53]:
from huggingface_hub import snapshot_download
from pathlib import Path

repo_id = "shishirgarde/gridpulse-ensemble-v1"

download_dir = Path("../downloads/gridpulse-ensemble-v1").resolve()

downloaded_path = snapshot_download(
    repo_id=repo_id,
    repo_type="model",
    revision="e2dcdfda9380130aa0c2624f4d7c641f0669c2d8",
    local_dir=str(download_dir),
)

print("Downloaded to:", downloaded_path)

for path in sorted(download_dir.rglob("*")):
    if path.is_file() and ".cache" not in path.parts:
        print(path.relative_to(download_dir))

Downloaded to: downloads\gridpulse-ensemble-v1
.gitattributes
chronos2_calendar_200\adapter_config.json
chronos2_calendar_200\adapter_model.safetensors
gridpulse\__init__.py
gridpulse\inference.py
gridpulse\tcn_model.py
gridpulse_ensemble_v1\metadata.json
gridpulse_ensemble_v1\tcn_checkpoint.pt
README.md
requirements.txt


In [54]:
import subprocess
import sys
import os
import json
import textwrap
import numpy as np

payload = {
    "origin": str(valid_origins[0].date()),
    "history": Xh[0].tolist(),
    "weather": weather_records,
}

script = textwrap.dedent("""
import json
import sys

from gridpulse.inference import GridPulseForecaster

request = json.loads(sys.stdin.read())
model = GridPulseForecaster.from_local(".")

result = model.predict(
    forecast_origin=request["origin"],
    demand_history_mu=request["history"],
    weather_forecast=request["weather"],
)

print("GRIDPULSE_RESULT=" + json.dumps([
    item["ensemble_mu"] for item in result["predictions"]
]))
""")

env = os.environ.copy()
env["PYTHONPATH"] = str(download_dir)

process = subprocess.run(
    [sys.executable, "-c", script],
    cwd=download_dir,
    env=env,
    input=json.dumps(payload),
    capture_output=True,
    text=True,
    encoding="utf-8",
    errors="replace",
    timeout=180,
)

if process.returncode != 0:
    print(process.stderr[-4000:])
    raise RuntimeError("Downloaded package inference failed")

line = next(
    x for x in process.stdout.splitlines()
    if x.startswith("GRIDPULSE_RESULT=")
)

downloaded_predictions = np.array(
    json.loads(line.split("=", 1)[1])
)

difference = np.max(
    np.abs(downloaded_predictions - ensemble_predictions[0])
)

print(f"Maximum difference: {difference:.8f} MU")

assert difference < 0.001
print("Hugging Face download and inference verified.")

Maximum difference: 0.00000763 MU
Hugging Face download and inference verified.


## Verify the deployed Modal endpoint and the standalone client

In [57]:
from dotenv import load_dotenv
from pathlib import Path
import requests
import os

load_dotenv(Path("../.env"), override=True)

URL = "https://shishir-garde--gridpulse-predict.us-east.modal.run"

api_payload = {
    "forecast_origin": payload["origin"],
    "demand_history_mu": payload["history"],
    "weather_forecast": payload["weather"],
}

response = requests.post(
    URL,
    headers={
        "Modal-Key": os.environ["MODAL_PROXY_KEY"],
        "Modal-Secret": os.environ["MODAL_PROXY_SECRET"],
    },
    json=api_payload,
    timeout=600,
)

print("HTTP status:", response.status_code)
response.raise_for_status()

result = response.json()
print("Predictions returned:", len(result["predictions"]))
print(result["predictions"][0])

HTTP status: 200
Predictions returned: 7
{'date': '2025-01-01', 'horizon': 1, 'chronos2_mu': 84.63080596923828, 'tcn_mu': 82.90045166015625, 'ensemble_mu': 83.765625}


In [58]:
import numpy as np

modal_predictions = np.array([
    row["ensemble_mu"]
    for row in result["predictions"]
])

reference = np.asarray(ensemble_predictions[0])

difference = np.max(np.abs(modal_predictions - reference))

print(f"Maximum difference: {difference:.8f} MU")

assert difference < 0.001, "Modal predictions differ from reference"

print("LIVE MODAL API VERIFIED")

Maximum difference: 0.00000763 MU
LIVE MODAL API VERIFIED


In [60]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path("../src/gridpulse").resolve()))

from gridpulse_client import GridPulseClient

client = GridPulseClient()

forecast = client.predict(
    forecast_origin=payload["origin"],
    demand_history_mu=payload["history"],
    weather_forecast=payload["weather"],
)

print("Forecast days:", len(forecast["predictions"]))

for row in forecast["predictions"]:
    print(
        row["date"],
        f"Chronos: {row['chronos2_mu']:.2f}",
        f"TCN: {row['tcn_mu']:.2f}",
        f"Ensemble: {row['ensemble_mu']:.2f} MU",
    )

Forecast days: 7
2025-01-01 Chronos: 84.63 TCN: 82.90 Ensemble: 83.77 MU
2025-01-02 Chronos: 84.67 TCN: 81.14 Ensemble: 82.91 MU
2025-01-03 Chronos: 84.93 TCN: 79.55 Ensemble: 82.24 MU
2025-01-04 Chronos: 82.81 TCN: 78.71 Ensemble: 80.76 MU
2025-01-05 Chronos: 81.42 TCN: 79.43 Ensemble: 80.42 MU
2025-01-06 Chronos: 84.40 TCN: 79.95 Ensemble: 82.18 MU
2025-01-07 Chronos: 85.19 TCN: 79.74 Ensemble: 82.47 MU


In [61]:
import numpy as np

client_predictions = np.array([
    row["ensemble_mu"]
    for row in forecast["predictions"]
])

difference = np.max(
    np.abs(client_predictions - ensemble_predictions[0])
)

print(f"Maximum difference: {difference:.8f} MU")

assert difference < 0.001

print("STANDALONE CLIENT VERIFIED")

Maximum difference: 0.00000763 MU
STANDALONE CLIENT VERIFIED
